# 03 · The 1D Heat Equation — Your First PDE

**Level:** Intermediate
**Prerequisites:** `01_intro_to_pinn`, `02_second_order_odes`

We now move from ODEs (one independent variable) to a **partial** differential
equation with both space *x* and time *t*. The heat/diffusion equation is the
canonical parabolic PDE and appears everywhere: thermal management, drug
diffusion, heat treatment of metals, battery thermal models.

### What you will learn
1. Handling a **2-input** network `(x, t) → u`
2. Mixing partial derivatives (`∂u/∂t`, `∂²u/∂x²`) via `autograd`
3. Imposing an **initial condition** and **two boundary conditions**
4. Validating against the analytical Fourier-series solution
5. Visualising the full space-time field

### References
- Raissi, Perdikaris & Karniadakis (2019), *J. Comput. Phys.* 378:686-707
- Lu et al. (2021), *DeepXDE*, SIAM Review 63(1):208-228 (library with many PDE examples)


## 1 · Problem statement

Solve the heat equation on a rod of length 1 with fixed cold ends:

$$\frac{\partial u}{\partial t} = \alpha\,\frac{\partial^2 u}{\partial x^2},
\qquad x\in[0,1],\; t\in[0,T]$$

with

$$u(x,0)=\sin(\pi x)\quad(\text{initial}),\qquad
u(0,t)=u(1,t)=0\quad(\text{boundary}).$$

The exact solution is a single decaying Fourier mode:

$$u(x,t)=\sin(\pi x)\,e^{-\alpha\pi^2 t}.$$


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

alpha = 0.4      # thermal diffusivity
T     = 1.0

def exact(x, t):
    return np.sin(np.pi*x) * np.exp(-alpha*np.pi**2*t)


In [ ]:
class PINN(nn.Module):
    def __init__(self, layers):
        super().__init__()
        seq = []
        for i in range(len(layers)-1):
            seq.append(nn.Linear(layers[i], layers[i+1]))
            if i < len(layers)-2:
                seq.append(nn.Tanh())
        self.net = nn.Sequential(*seq)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight); nn.init.zeros_(m.bias)
    def forward(self, x, t):
        return self.net(torch.cat([x, t], dim=1))

model = PINN([2, 64, 64, 64, 1]).to(device)   # input (x,t) -> output u
print("params:", sum(p.numel() for p in model.parameters()))


In [ ]:
# --- Sample points ---
N_f  = 4000    # interior collocation (PDE residual)
N_ic = 200     # initial condition
N_bc = 200     # each boundary

def sample():
    x_f = torch.rand(N_f, 1, device=device)
    t_f = torch.rand(N_f, 1, device=device) * T
    x_f.requires_grad_(True); t_f.requires_grad_(True)

    x_ic = torch.rand(N_ic, 1, device=device)
    t_ic = torch.zeros(N_ic, 1, device=device)
    u_ic = torch.sin(np.pi * x_ic)

    t_bc = torch.rand(N_bc, 1, device=device) * T
    x_b0 = torch.zeros(N_bc, 1, device=device)
    x_b1 = torch.ones(N_bc, 1, device=device)
    return x_f, t_f, x_ic, t_ic, u_ic, x_b0, x_b1, t_bc


In [ ]:
def pde_residual(model, x, t):
    u   = model(x, t)
    u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
    u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
    u_xx= torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]
    return u_t - alpha * u_xx

def loss_fn(model, pts):
    x_f, t_f, x_ic, t_ic, u_ic, x_b0, x_b1, t_bc = pts
    r = pde_residual(model, x_f, t_f)
    L_pde = torch.mean(r**2)
    L_ic  = torch.mean((model(x_ic, t_ic) - u_ic)**2)
    L_bc  = torch.mean(model(x_b0, t_bc)**2) + torch.mean(model(x_b1, t_bc)**2)
    return L_pde + L_ic + L_bc, (L_pde.item(), L_ic.item(), L_bc.item())


In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
EPOCHS = 12000
pts = sample()
hist = []
for ep in range(EPOCHS):
    optimizer.zero_grad()
    loss, parts = loss_fn(model, pts)
    loss.backward(); optimizer.step()
    hist.append(loss.item())
    if ep % 2000 == 0:
        print(f"epoch {ep:5d} | loss {loss.item():.3e} | pde {parts[0]:.2e} ic {parts[1]:.2e} bc {parts[2]:.2e}")
print("training complete")


In [ ]:
# --- Evaluate on a grid ---
nx, nt = 100, 100
xs = np.linspace(0, 1, nx); ts = np.linspace(0, T, nt)
X, Tg = np.meshgrid(xs, ts)
xf = torch.tensor(X.ravel(), dtype=torch.float32, device=device).view(-1,1)
tf = torch.tensor(Tg.ravel(), dtype=torch.float32, device=device).view(-1,1)
with torch.no_grad():
    U = model(xf, tf).cpu().numpy().reshape(nt, nx)
Ue = exact(X, Tg)
relL2 = np.linalg.norm(U-Ue)/np.linalg.norm(Ue)
print(f"relative L2 error over space-time: {relL2:.3e}")

fig, ax = plt.subplots(1, 3, figsize=(15,4))
im0 = ax[0].pcolormesh(X, Tg, U, shading='auto'); ax[0].set_title('PINN u(x,t)')
im1 = ax[1].pcolormesh(X, Tg, Ue, shading='auto'); ax[1].set_title('Exact')
im2 = ax[2].pcolormesh(X, Tg, np.abs(U-Ue), shading='auto'); ax[2].set_title('|error|')
for a, im in zip(ax, [im0,im1,im2]):
    a.set_xlabel('x'); a.set_ylabel('t'); fig.colorbar(im, ax=a)
plt.tight_layout(); plt.show()


In [ ]:
# --- Snapshots at several times ---
plt.figure(figsize=(7,4))
for tv in [0.0, 0.15, 0.4, 0.8]:
    idx = np.argmin(np.abs(ts - tv))
    plt.plot(xs, U[idx], lw=2, label=f'PINN t={ts[idx]:.2f}')
    plt.plot(xs, exact(xs, ts[idx]), 'k--', lw=1)
plt.xlabel('x'); plt.ylabel('u'); plt.legend(); plt.title('Temperature profile decaying over time')
plt.tight_layout(); plt.show()


## 2 · What changed vs the ODE notebooks?

| Aspect | ODE (nb 01/02) | PDE (this notebook) |
|---|---|---|
| Network input | `t` (1-D) | `(x, t)` (2-D) |
| Derivatives | `du/dt`, `d²u/dt²` | mixed: `∂u/∂t`, `∂²u/∂x²` |
| Constraints | initial condition(s) | initial **field** + boundary conditions |
| Collocation | points on a line | points in a 2-D domain |

The residual pattern is identical — we just take the partial derivatives we need
and drive their combination to zero.
